# Feature enrichment: from a set of sequences to an RL target

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rotskoff-group/idiom/blob/main/cookbook/notebooks/feature_enrichment.ipynb)

Give this notebook a set of sequences you care about — a compartment, a functional class, hits from
a screen — and it finds which SAE features fire in them far more often than in a background, writes
the strongest as a **signature**, and shows the residue grammar behind each one.

The signature is directly consumable by the `sae_only_<name>` reward, so the end of this notebook is
the command that designs new sequences carrying the same feature code.

**A GPU is strongly recommended**: the cost is dominated by encoding the background.

> **Runtime → Change runtime type → GPU** in Colab before running.

In [ ]:
# Install IDiom if it is not already available (Colab, or a fresh environment).
# Takes a couple of minutes the first time; it pulls lightning, hydra and wandb too.
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("idiom") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/rotskoff-group/idiom.git"])

from huggingface_hub import hf_hub_download

import idiom


def example_data(name: str) -> str:
    """Download one cookbook example file from the Hub and return its local path."""
    return hf_hub_download("jxliu2/idiom-data", f"example_data/{name}", repo_type="dataset")


print("idiom", idiom.__file__)

## Parameters

`MAX_BACKGROUND` dominates the runtime. 10,000 is what the shipped signatures were built with; drop
it to ~2,000 for a quick look, or if you are on CPU.

In [ ]:
POSITIVE = example_data("protgps/nucleolus.fasta")  # the set you care about
NAME = "nucleolus"          # signature name; the reward becomes sae_only_<name>
SAE = "jxliu2/idiomsae-300M-L18-k32"
DEVICE = "auto"
OUT = "enr"                 # working directory for the feature datasets
SIGNATURE = "signature.json"  # what the RL run reads
MAX_BACKGROUND = 10000      # background sequences to encode -- this dominates runtime
TOP_N = 30                  # features kept in the signature
CASE = "top30"              # case name to store the signature under
SEED = 0
N_FEATURES = 6              # features to logo
N_WINDOWS = 60              # top windows stacked per feature
HALF_WIDTH = 7              # residues each side of the peak

In [ ]:
from pathlib import Path

import numpy as np

from idiom import IDiomSAE
from idiom.sae.features import per_sequence_activations, top_windows
from idiom.sae.features.enrichment import (
    FDR_ALPHA,
    LOG2OR_FLOOR,
    enrich,
    enriched_mask,
    feature_counts,
    length_match,
    load_sequences,
    top_features,
    write_signature,
)

out = Path(OUT)
out.mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(SEED)

sae = IDiomSAE.from_pretrained(SAE, device=DEVICE)
print(f"SAE: layer {sae.layer} of {sae.host_model}, {sae.sae.num_latents} latents")

## The two sets

`load_sequences` reads a FASTA whether or not its headers carry an IDiom `_IDR_x-y` span; without
one, the whole sequence is treated as the IDR.

In [ ]:
positives = load_sequences(POSITIVE)
print(f"positive set: {len(positives)} sequences from {POSITIVE}")

The background defaults to the held-out validation split of the pretraining corpus (122 MB).

It is sampled **length-matched** to the positive set: features that merely track length look
enriched when the two sets have different length distributions, which they usually do.

In [ ]:
bg_path = hf_hub_download("jxliu2/idiom-data", "training_sequences/validation.fasta",
                          repo_type="dataset")
background = load_sequences(bg_path)
print(f"background pool: {len(background)} sequences")

background = length_match(positives, background, n=MAX_BACKGROUND, rng=rng)
print(f"sampled: {len(background)} length-matched to the positive set")

## Encode both sets through the SAE

This is the slow part — the background is the bulk of it.

In [ ]:
pos_fd = sae.build_feature_dataset(positives, out / "fd_positive", batch_size=16)
bg_fd = sae.build_feature_dataset(background, out / "fd_background", batch_size=16)
print("done")

## Enrichment

A feature "fires" in a sequence if the SAE selects it at any residue, counted once per sequence.
From the two firing counts, `enrich` computes a Haldane-Anscombe log2 odds ratio, standardizes it
against a hypergeometric null, and controls the false discovery rate with Benjamini-Hochberg.

In [ ]:
a, n_pos = feature_counts(pos_fd)
b, n_neg = feature_counts(bg_fd)
result = enrich(a, n_pos, b, n_neg, sae.sae.num_latents)
mask = enriched_mask(result)

print(f"{int(mask.sum())} enriched features")
print(f"  FDR < {FDR_ALPHA}, log2 odds ratio >= {LOG2OR_FLOOR}, prevalence >= 5%")

In [ ]:
import matplotlib.pyplot as plt

active = result["active"]
x, y = result["log2or"][active], np.abs(result["z"][active])
enr = mask[active]

fig, ax = plt.subplots(figsize=(4.6, 3.6), constrained_layout=True)
ax.scatter(x[~enr], y[~enr], s=4, alpha=0.25, lw=0, color="#c3ced0", label="other")
ax.scatter(x[enr], y[enr], s=8, alpha=0.9, lw=0, color="#c1440e", label="enriched")
sig = result["fdr"][active] < FDR_ALPHA
if sig.any():                                  # the L-shaped enriched boundary
    ax.axhline(float(y[sig].min()), ls="--", lw=0.8, color="#110d1b")
    ax.axvline(LOG2OR_FLOOR, ls="--", lw=0.8, color="#110d1b")
ax.axvline(0, lw=0.8, color="#110d1b")
ax.set_xlabel("log$_2$ odds ratio")
ax.set_ylabel("|z|")
ax.set_title(f"{NAME}: {int(mask.sum())} enriched features", fontsize=10)
ax.legend(loc="upper left", fontsize=8, frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.show()

## The signature

`top_features` keeps the strongest by odds ratio and drops **boundary features** — ones whose
firings sit at an IDR's first or last residues, which detect the excision point rather than a
motif.

In [ ]:
ids = top_features(result, n=TOP_N, drop_boundary=True, feature_dir=bg_fd)
sig_path = write_signature(
    SIGNATURE, {NAME: ids}, case=CASE,
    provenance={"sae": SAE, "positive": POSITIVE, "background": str(bg_path),
                "n_pos": int(n_pos), "n_background": int(n_neg),
                "length_matched": True, "boundary_dropped": True,
                "rank": "log2 odds ratio, descending"})

print(f"wrote {sig_path}")
print(f"{len(ids)} features: {ids[:8]}{' ...' if len(ids) > 8 else ''}")

## What the enriched features detect

For each feature, take the windows where it fires hardest across the positive set, stack them, and
render an information-content logo. This is the grammar the signature is made of.

In [ ]:
import logomaker

feats, index = sae.encode(POSITIVE, pool="none")
per_seq = per_sequence_activations(feats, index)
print(f"{feats.shape[0]} residues across {len(per_seq)} sequences")

show = ids[:N_FEATURES]
fig, axes = plt.subplots(len(show), 1, figsize=(6, 1.5 * len(show)),
                         constrained_layout=True, squeeze=False)
for ax, fid in zip(axes[:, 0], show):
    windows = top_windows(fid, feats, per_seq, n_windows=N_WINDOWS, half_width=HALF_WIDTH)
    if len(windows) < 2:
        ax.set_axis_off()
        ax.set_title(f"feature {fid}: too few windows", fontsize=8)
        continue
    logomaker.Logo(logomaker.alignment_to_matrix(windows, to_type="information"),
                   ax=ax, color_scheme="chemistry")
    ax.set_title(f"feature {fid}  ({len(windows)} windows)", fontsize=8)
    ax.set_ylabel("bits", fontsize=7)
    ax.set_xticks([])
plt.show()

## Design new sequences carrying this code

The signature file is what the `sae_only_<name>` reward reads. Point the two environment variables
at it and append the term to the guardrails `grpo.yaml` already carries.

This is a **real training run** — 1 GPU for hours — so it belongs on a cluster, not in this
notebook. Download `signature.json` and run the command below, or use
`cookbook/scripts/grpo/sae_features.bash`, which sets the same thing.

In [ ]:
term = f"{{reward: sae_only_{NAME}, module: idiom.train.grpo.reward.sae_feature, weight: 1.0}}"
print(f"IDIOM_SAEREWARD_FEATURES={Path(sig_path).resolve()} IDIOM_SAEREWARD_CASE={CASE} \\")
print("  idiom_train_grpo init_from=jxliu2/idiom-300M \\")
print(f"    reward.add='[{term}]'")